In [1]:
from primaite.agents.aegis.gllm import GLLM
from primaite.agents.aegis.modules.openai import OpenAIClient
from primaite.agents.git_agent import GITAgent
from torch.utils.data import DataLoader
from primaite.agents.llm.utils import network_connectivity_desc
import logging

logging.disable(logging.CRITICAL)

/home/jonathan/projects/primaite/PrimAITE/aegis/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/jonathan/projects/primaite/PrimAITE/aegis/lib/python3.10/site-packages/pydantic/_internal/_fields.py:161: UserWarning: Field "model_id" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
2024-08-29 08:53:23.295720: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2024-08-29 08:53:23.342577: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow wit

In [2]:
gllm = GLLM()
openai = OpenAIClient(openai_api_key="")

In [3]:
gllm.llm.tokenizer.apply_chat_template(
    [
        {"role": "system", "content": "You are a helpful assistant"},
        {"role": "user", "content": "hello LLM!"},
        {"role": "assistant", "content": "Hi there!"},
    ],
    tokenize=False,
)

'<|im_start|>system\nYou are a helpful assistant<|im_end|>\n<|im_start|>user\nhello LLM!<|im_end|>\n<|im_start|>assistant\nHi there!<|im_end|>\n'

In [4]:
questions = [
    "What is the total number of nodes in the network?",
]

In [5]:
# Mock a primaite graph for development
agent = GITAgent(
    training_config_path="../src/primaite/config/_package_data/training/git.yaml",
    lay_down_config_path="../src/primaite/config/_package_data/lay_down/lay_down_config_6_data_manipulation.yaml",
)
obs = agent._env.reset()
data = agent.create_graph(obs)
network_desc = network_connectivity_desc(agent._env)

<Figure size 640x480 with 0 Axes>

In [6]:
agent._env.network

In [7]:
import os
import pickle as pkl

if "openai_responses.pkl" not in os.listdir("./"):
    openai_responses = []
    openai_prompts = gllm.build_prompts(questions=questions, network_desc=network_desc, model="openai")
    for prompt in openai_prompts:
        openai_responses.append(openai.generate(prompt=prompt))

    with open("openai_responses.pkl", "wb") as file:
        pkl.dump(openai_responses, file)
else:
    with open("openai_responses.pkl", "rb") as file:
        openai_responses = pkl.load(file)

In [8]:
openai_responses

['There are a total of 9 nodes in the network.']

In [9]:
openai_responses

['There are a total of 9 nodes in the network.']

In [10]:
data

DataBatch(edge_index=[2, 18], id=[1], num_nodes=9, x=[9, 6], batch=[9], ptr=[2])

In [11]:
from primaite.agents.aegis.data import GLLMDataset, collate_fn

dataset = GLLMDataset(
    graphs=[data for _ in range(len(questions))],
    questions=[question for question in questions],
    gt_answers=[response for response in openai_responses],
    llm=gllm.llm,
)

target gt probs shape: torch.Size([1, 12, 49153])


In [12]:
dataloader = DataLoader(dataset, batch_size=10, shuffle=True, collate_fn=collate_fn)

In [13]:
gllm.llm.tokenizer.pad_token

'<|pad|>'

In [14]:
from primaite.agents.aegis.gllm import train_loop

gllm_responses = train_loop(
    model=gllm, dataloader=dataloader, network_desc=network_desc, n_epochs=100, loss_fn="crossentropy"
)

/home/jonathan/projects/primaite/PrimAITE/aegis/lib/python3.10/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]


Epoch 0 Loss: 9.78676700592041
There are a total of 9 nodes in the network. <|im_end|>
The total number of nodes in the network is the sum of the number of nodes in each of the two sub-networks.<|im_end|>
Epoch 1 Loss: 5.9233078956604
There are a total of 9 nodes in the network. <|im_end|>
What are the number of the most popular and well-knownized characters in the network.<|im_end|>
Epoch 2 Loss: 5.096622943878174
There are a total of 9 nodes in the network. <|im_end|>
What is the network of, or the network, or the network, or the network.

Question:

What is the network of, or the network, or the network, or the network, or the network, or the network, or the network, or the network, or the network, or the network, or the network, or the network, or the network, or the network, or the network, or the network, or the network, or the network, or the
Epoch 3 Loss: 4.260982036590576
There are a total of 9 nodes in the network. <|im_end|>
What is the total number of nodes in the network?<

# Training notes
- Truncate GLLM response at GT max length to teach GLLM when it should have stopped generating.
- GLLM cosine embs loss function broke - fix so we can show that it wasn't viable in the report.